# Gold Processing — Customer

Enriches `ecommerce.silver.silver_customers` with geographic regions into `ecommerce.gold.gold_customer`:

* All `silver_customers` columns pass through: `customer_id`, `phone`, `country_code`, `country`, `state`, `source_file`, `ingested_at`
* **`region`** pulled from `ecommerce.silver.silver_region` (join on `country_code` + `state`) and placed beside `state`
* Coverage is guaranteed: `silver_region` was built from exactly the pairs present in `silver_customers` (validated in `dim_silver_6`)

In [0]:
from pyspark.sql.functions import col

# Build the enriched gold DataFrame and write it
target_table = "ecommerce.gold.gold_customer"

df_gold_customer = (
    spark.table("ecommerce.silver.silver_customers")
    .join(
        spark.table("ecommerce.silver.silver_region").select("country_code", "state", "region"),
        on=["country_code", "state"],
        how="left",
    )
    # All silver_customers columns plus region beside state
    .select(
        "customer_id",
        "phone",
        "country_code",
        "country",
        "state",
        "region",
        "source_file",
        "ingested_at",
    )
)

unmatched = df_gold_customer.where(col("region").isNull()).count()
print(f"Rows: {df_gold_customer.count()} | customers without a region: {unmatched}")
assert unmatched == 0, "Some customers did not match a region"

# Idempotent full load into the gold table
(
    df_gold_customer.write
    .mode("overwrite")
    .saveAsTable(target_table)
)

print(f"Wrote {spark.table(target_table).count()} rows to {target_table}")

In [0]:
# Verify the persisted gold table
spark.sql("""
    SELECT
      COUNT(*) AS rows,
      COUNT_IF(region IS NULL) AS null_regions,
      COUNT(DISTINCT region) AS distinct_regions
    FROM ecommerce.gold.gold_customer
""").show()

spark.sql("""
    SELECT region, COUNT(*) AS customers
    FROM ecommerce.gold.gold_customer
    GROUP BY region
    ORDER BY customers DESC
""").show(20, truncate=False)

display(spark.table("ecommerce.gold.gold_customer").limit(5))